In [0]:
%sql

CREATE OR REPLACE TABLE `p-cdh-delivery`.it.master_slaves AS
WITH portfolios_scope AS (
    SELECT 'Orion X Securitisation DAC' AS PortfolioName UNION ALL
    SELECT 'Galaxy II Funding DAC' UNION ALL
    SELECT 'Cosmos Retail'
),
products_scope AS (
    SELECT '40011' AS ProductCode UNION ALL
    SELECT '40569' UNION ALL
    SELECT '40584' UNION ALL
    SELECT '40585' UNION ALL
    SELECT '41585' UNION ALL
    SELECT '40586' UNION ALL
    SELECT '41285'
),
ref_date AS (
    SELECT MAX(BusinessDate) AS Ref_Date 
    FROM derivative.dexposure
),
loans_us AS (
    SELECT
      PortfolioName,
      CASE 
        WHEN derivative.is_valid_afm(PartyTaxRegistrationNumber) 
             THEN PartyTaxRegistrationNumber 
        ELSE CDI 
      END AS Tax_ID,
      CDI,
      AgreementKey,
      ExposureKey,
      Opendate,
      ExposureStatus,
      derivative.converttoeur(currency, OutstandingTotalBalance, BusinessDate) AS TLC_LCY,
      derivative.converttoeur(currency, ConditionalWriteOffAmount, BusinessDate) AS CWOFF_LCY,
      ConditionalWriteOffDate,
      ExposureRestructureType,
      ExposureRestructureDate,
      ExposureRestructureOrigination,
      BusinessDate,
      MaturityDate,
      NextInstallmentDate,
      NextInstallmentCapitalDate
    FROM derivative.dexposure
    WHERE IsActiveFlag = TRUE
      AND SourceSystem = 'profits'
      AND BusinessDate = (SELECT Ref_Date FROM ref_date)
      AND PortfolioName IN (SELECT PortfolioName FROM portfolios_scope)
      AND ProductCode IN (SELECT ProductCode FROM products_scope)
),
master_slaves AS (
    SELECT 
        A.PortfolioName,
        A.Tax_ID,
        A.CDI,
        A.AgreementKey,
        A.ExposureKey AS Master,
        B.ExposureKey AS Slave,
        A.BusinessDate,
        A.Opendate AS Master_Open_Date,
        A.TLC_LCY AS Master_TLC,
        B.CWOFF_LCY AS Slave_CWOFF,
        B.MaturityDate AS Slave_Maturity,
        A.ExposureStatus AS Master_Status,
        B.ExposureStatus AS Slave_Status,
        ROW_NUMBER() OVER (
            PARTITION BY A.ExposureKey
            ORDER BY B.MaturityDate DESC, B.CWOFF_LCY DESC
        ) AS Slave_RN
    FROM loans_us A
    JOIN loans_us B
      ON A.AgreementKey = B.AgreementKey
     AND A.ExposureKey <> B.ExposureKey
    WHERE (
            A.ExposureRestructureType = 'Restructure'
        AND (A.CWOFF_LCY IS NULL OR A.CWOFF_LCY = 0)
        AND A.ExposureRestructureOrigination = 'Cepal Restructure'
        AND B.ExposureRestructureType = 'Conditional Woff'
        AND B.CWOFF_LCY > 0
        AND B.ExposureRestructureOrigination IS NULL
        AND A.ExposureStatus IN ('Normal', 'Overdue')
        )
    OR (
            (A.CWOFF_LCY IS NULL OR A.CWOFF_LCY = 0)
        AND B.CWOFF_LCY > 0
        AND A.ExposureStatus IN ('Normal', 'Overdue')
        )
)
SELECT 
    master, 
    slave, 
    businessdate
FROM master_slaves
WHERE Slave_RN = 1
ORDER BY PortfolioName, Tax_ID, CDI, AgreementKey, Master, Slave_RN;
